# 03 - Ground truth oracional y evaluacion causal

Este notebook corrige una simplificacion importante: el cierre no se evalua por clip aislado.

La pregunta real es: dada una secuencia de clips de una misma fuente, cuanto texto acumula el sistema antes de decidir que termino una oracion.

## 1. Setup

No se abren videos, ROIs `.npz`, checkpoints ni outputs de entrenamiento. Se usan splits livianos y un ejemplo chico de ground truth oracional.

In [1]:
from pathlib import Path
import csv
import json
import sys
from collections import Counter
from IPython.display import Markdown, display

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "realtime" / "src").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def _fmt(value):
    if isinstance(value, float):
        return f"{value:.4f}"
    if isinstance(value, (list, tuple)):
        return ", ".join(str(item) for item in value) or "-"
    text = str(value)
    return text.replace("\n", " ").replace("|", "\\|")


def show_table(rows, columns):
    if not rows:
        display(Markdown("_Sin filas para mostrar._"))
        return
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(_fmt(row.get(col, "")) for col in columns) + " |")
    display(Markdown("\n".join([header, sep, *body])))

print(f"Repo detectado: {ROOT.name}")
from realtime.src.provider_factory import make_closure_provider
from realtime.src.secuencias import (
    evaluate_causal_sequence,
    export_llm_annotation_packet,
    load_clips_from_split,
    load_sequence_ground_truth,
)

split_path = ROOT / "vsr_models" / "splits" / "val.csv"
ground_truth_path = ROOT / "realtime" / "examples" / "ground_truth_demo.json"
out_dir = ROOT / "realtime" / "outputs" / "notebooks" / "03_ground_truth"
out_dir.mkdir(parents=True, exist_ok=True)
annotation_packet_path = out_dir / "annotation_packet_demo.md"

print("Split:", split_path.relative_to(ROOT))
print("Ground truth demo:", ground_truth_path.relative_to(ROOT))
print("Output de anotacion:", annotation_packet_path.relative_to(ROOT))

Repo detectado: labios-argentos
Split: vsr_models\splits\val.csv
Ground truth demo: realtime\examples\ground_truth_demo.json
Output de anotacion: realtime\outputs\notebooks\03_ground_truth\annotation_packet_demo.md


## 2. Que le pasamos a un LLM fuerte

Para anotar una fuente real, primero exportamos clips ordenados. Ese paquete se puede pegar en GPT/Claude con el prompt de `GROUND_TRUTH_ORACIONAL.md`.

La salida esperada del LLM no es la evaluacion final: es una preanotacion que conviene revisar manualmente.

In [2]:
with split_path.open("r", encoding="utf-8", newline="") as fh:
    first_rows = list(csv.DictReader(fh))[:5]
source_id = first_rows[0]["titulo"]
clips_for_annotation = load_clips_from_split(split_path, source_id=source_id, limit=5)
export_llm_annotation_packet(clips_for_annotation, source_id=source_id, output_path=annotation_packet_path)

show_table(
    [
        {
            "order": clip.order,
            "clip_id": clip.clip_id,
            "texto": clip.text,
            "n_frames": clip.n_frames,
        }
        for clip in clips_for_annotation
    ],
    ["order", "clip_id", "texto", "n_frames"],
)
print("Paquete exportado:", annotation_packet_path.relative_to(ROOT))

| order | clip_id | texto | n_frames |
| --- | --- | --- | --- |
| 1 | clip_0000 | es una verguenza absoluta lo que acaba de pasar en la cancha de racing | 128 |
| 2 | clip_0001 | y la verdad es que cualquier cosa que pueda decir en este video | 139 |
| 3 | clip_0002 | no no me voy a arrepentir no no me voy a arrepentir bajo ningun concepto | 157 |
| 4 | clip_0003 | porque tengo los huevos llenos de este equipo de racing y la realidad es que no hay manera de que el tecnico pueda seguir despues de quedar afuera | 264 |
| 5 | clip_0006 | ahora no sali en ni segundo es una cosa inaudita | 105 |

Paquete exportado: realtime\outputs\notebooks\03_ground_truth\annotation_packet_demo.md


## 3. Formato de ground truth que vamos a usar

Cada oracion indica que clips cubre y despues de que clip un sistema causal deberia commitearla.

In [3]:
sequence = load_sequence_ground_truth(ground_truth_path)
show_table(
    [
        {
            "sentence_id": sentence.sentence_id,
            "text": sentence.text,
            "start_clip": sentence.start_clip,
            "end_clip": sentence.end_clip,
            "commit_after_clip": sentence.commit_after_clip,
        }
        for sentence in sequence.sentences
    ],
    ["sentence_id", "text", "start_clip", "end_clip", "commit_after_clip"],
)

| sentence_id | text | start_clip | end_clip | commit_after_clip |
| --- | --- | --- | --- | --- |
| s001 | Yo creo que este modulo puede funcionar bastante bien. | clip_0000 | clip_0001 | clip_0001 |
| s002 | Si lo evaluamos con cuidado, despues podemos comparar contra un LLM. | clip_0002 | clip_0003 | clip_0003 |

## 4. Timeline causal: clips, buffer y decision esperada

Esta es la visualizacion importante. El sistema no ve el futuro: en cada fila solo tiene el buffer acumulado hasta el clip actual.

In [4]:
expected_by_clip = {sentence.commit_after_clip: sentence.sentence_id for sentence in sequence.sentences}
buffer = []
timeline = []
for clip in sequence.clips:
    buffer.append(clip.text)
    expected_action = "commit" if clip.clip_id in expected_by_clip else "wait"
    timeline.append({
        "clip": clip.clip_id,
        "texto_clip": clip.text,
        "buffer_visible": " ".join(buffer),
        "esperado": expected_action,
        "sentence_id": expected_by_clip.get(clip.clip_id, ""),
    })
    if expected_action == "commit":
        buffer = []

show_table(timeline, ["clip", "texto_clip", "buffer_visible", "esperado", "sentence_id"])

| clip | texto_clip | buffer_visible | esperado | sentence_id |
| --- | --- | --- | --- | --- |
| clip_0000 | yo creo que este modulo | yo creo que este modulo | wait |  |
| clip_0001 | puede funcionar bastante bien | yo creo que este modulo puede funcionar bastante bien | commit | s001 |
| clip_0002 | si lo evaluamos con cuidado | si lo evaluamos con cuidado | wait |  |
| clip_0003 | despues podemos comparar contra un llm | si lo evaluamos con cuidado despues podemos comparar contra un llm | commit | s002 |

## 5. Evaluacion causal de la baseline

Aca recien tienen sentido las metricas de cierre: no medimos si un clip individual parece largo, sino si el commit cae en el punto esperado de la secuencia.

In [5]:
summary = evaluate_causal_sequence(sequence, make_closure_provider("heuristic"))
metric_rows = [
    {"metrica": "clips", "valor": summary["clips"]},
    {"metrica": "commits_esperados", "valor": summary["expected_commits"]},
    {"metrica": "commits_predichos", "valor": summary["predicted_commits"]},
    {"metrica": "commits_correctos", "valor": summary["correct_commits"]},
    {"metrica": "commits_tempranos", "valor": summary["early_commits"]},
    {"metrica": "waits_tardios", "valor": summary["late_waits"]},
    {"metrica": "commits_faltantes", "valor": summary["missing_commits"]},
    {"metrica": "precision_commit", "valor": summary["commit_precision"]},
    {"metrica": "recall_commit", "valor": summary["commit_recall"]},
    {"metrica": "latencia_p50_ms", "valor": summary["latency_ms"]["p50"]},
    {"metrica": "latencia_p95_ms", "valor": summary["latency_ms"]["p95"]},
]
show_table(metric_rows, ["metrica", "valor"])

| metrica | valor |
| --- | --- |
| clips | 4 |
| commits_esperados | 2 |
| commits_predichos | 2 |
| commits_correctos | 2 |
| commits_tempranos | 0 |
| waits_tardios | 0 |
| commits_faltantes | 0 |
| precision_commit | 1.0000 |
| recall_commit | 1.0000 |
| latencia_p50_ms | 0.0568 |
| latencia_p95_ms | 0.1320 |

## 6. Decisiones fila por fila

Si algo sale mal, esta tabla muestra si el error fue commit temprano, wait tardio o baja confianza inesperada.

In [6]:
rows = []
for row in summary["rows"]:
    rows.append({
        "clip": row["clip_id"],
        "buffer_desde": row["buffer_start_clip"],
        "esperado": row["expected_action"],
        "predicho": row["predicted_action"],
        "sentence_id": row["expected_sentence_id"],
        "razon": row["reason"],
        "riesgos": row["risk_flags"],
    })
show_table(rows, ["clip", "buffer_desde", "esperado", "predicho", "sentence_id", "razon", "riesgos"])

| clip | buffer_desde | esperado | predicho | sentence_id | razon | riesgos |
| --- | --- | --- | --- | --- | --- | --- |
| clip_0000 | clip_0000 | wait | wait |  | contexto_insuficiente | conservative_wait |
| clip_0001 | clip_0000 | commit | commit | s001 | contexto_suficiente_sin_conector_colgante | heuristic_commit |
| clip_0002 | clip_0002 | wait | wait |  | contexto_insuficiente | conservative_wait |
| clip_0003 | clip_0002 | commit | commit | s002 | contexto_suficiente_sin_conector_colgante | heuristic_commit |

## 7. Lectura final

Resultado de este notebook:

- La evaluacion seria de cierre necesita ground truth oracional, no solo casos sinteticos.
- Ya hay un formato concreto para que un LLM fuerte preanote una fuente y una persona la revise.
- El evaluador causal mide commits tempranos, waits tardios y commits faltantes.
- El modo offline/bidireccional puede agregarse despues, pero sus metricas deben separarse del modo causal.